In [1]:
import os

In [2]:
import os,json,time
from dotenv import load_dotenv
load_dotenv()

page_index_api_key=os.getenv("PAGEINDEX_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")

if not groq_api_key and page_index_api_key:
    print("api key not found ")
else:
    print("done api key found")

done api key found


In [3]:
import sys
print(sys.executable)

c:\Users\rahul\Downloads\Ai Roadmap 2026 pratu\rag_ai\.venv\Scripts\python.exe


In [7]:
from pageindex import PageIndexClient
from groq import Groq

pi_client=PageIndexClient(api_key=page_index_api_key)
groq_client=Groq(api_key=groq_api_key)

## section 2 upload and index pdf

In [9]:
# upload yor pdf

PDF_PATH="ai_radiology_sample.pdf"

print(f"Uploading :{PDF_PATH}")
result=pi_client.submit_document(PDF_PATH)

doc_id=result["doc_id"]

print("Uploaded sucessfully")
print(f"Document Id:{doc_id}")
print(" Save this Id -you ill use througout the notebook")



Uploading :ai_radiology_sample.pdf
Uploaded sucessfully
Document Id:pi-cmudrk8bw00000bnvmm01weuo
 Save this Id -you ill use througout the notebook


In [10]:
# build the tree

print("Building the tree index..")
print("these runs once ")

while True:
    status_result=pi_client.get_document(doc_id)
    status=status_result.get("status")

    print(f"Status:{status}")

    if status == "completed":
        print("tree index is ready")
        break
    elif status == "failed":
        print("processing failed")
        break

    time.sleep(5)

    

Building the tree index..
these runs once 
Status:completed
tree index is ready


## 🔍 Section 3: Inspect the Tree Structure
What the tree looks like:

Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
Each node has:

node_id — unique ID used during retrieval
title — section heading
page_index — page number in original PDF
text — section summary (when node_summary=True)
nodes — child sections (nested)

In [11]:
# fetch the full tree

tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get("result",[])

print(f"Top level section:{len(pageindex_tree)}")
print("\n raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {},indent=2))

Top level section:12

 raw tree (first node):
{
  "title": "1. Introduction to AI in Radiology",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This text introduces the role of artificial intelligence in radiology as a supportive tool for analyzing medical images. It outlines the standard AI radiology workflow\u2014from image acquisition to human review\u2014and notes the document's purpose as an educational sample for testing retrieval and question-answering systems.",
  "text": "# 1. Introduction to AI in Radiology\n\nArtificial intelligence (AI) is increasingly used as a supporting technology in medical imaging. Radiology produces large volumes of images, including X-rays, computed tomography (CT), magnetic resonance imaging (MRI), and ultrasound studies.\n\nAI systems can process image data and identify patterns that may be difficult to detect consistently. In clinical practice, these systems are generally intended to assist trained healthcare professionals rather than repla

In [13]:
# prety print

# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] 1. Introduction to AI in Radiology  (p.1)
[0001] 2. Medical Imaging Modalities  (p.2)
[0002] 3. X-ray Imaging  (p.3)
[0003] 4. CT and MRI Analysis  (p.4)
[0004] 5. AI Image Classification  (p.5)
[0005] 6. Convolutional Neural Networks  (p.6)
[0006] 7. Dataset Preparation  (p.7)
[0007] 8. Training and Evaluation  (p.8)
[0008] 9. Retrieval-Augmented Generation  (p.9)
[0009] 10. AI-Assisted Radiology Workflow  (p.10)
[0010] 11. Challenges and Limitations  (p.11)
[0011] 12. Conclusion  (p.12)


In [14]:
# count the total no of node

def count_nodes(nodes):
    total=len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total=count_nodes(pageindex_tree)
print(f"total nodes in tree {total}")
print("Each node =one retrievable section of the document ")



total nodes in tree 12
Each node =one retrievable section of the document 


##  🧠 Section 4: LLM Tree Search — The Core of PageIndex
This is where PageIndex fundamentally differs from vector RAG.

Vector RAG retrieval:
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
Problem: finds what's similar, not what's relevant

PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [15]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [16]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What are the main medical imaging modalities?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What are the main medical imaging modalities?

🧠 LLM Reasoning:
The query asks for the main medical imaging modalities. The document tree includes a section explicitly titled '2. Medical Imaging Modalities' (node_id 0001) which is most likely to list the primary modalities. While other nodes (X-ray Imaging, CT and MRI Analysis) cover specific modalities, the overarching list should be in node 0001, so that is the primary node to retrieve.

🎯 Selected Node IDs: ['0001']



## ⚙️ Section 5: Full End-to-End RAG Pipeline
3 steps:

Tree Search → LLM picks relevant node_ids
Retrieve → Fetch the actual section content from those nodes
Generate → LLM writes a grounded answer with page citations
What makes this better than vector RAG:

Retrieved content has titles + page numbers (traceable)
LLM can cite exactly which section the answer comes from
No hallucination from irrelevant chunks

In [17]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [18]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [19]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [20]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is patient-level separation in a medical dataset?",
    tree=pageindex_tree
)

🔍 Query: What is patient-level separation in a medical dataset?

🧠 Reasoning: The query asks for a definition of 'patient-level separation' in a medical dataset, which is a concept related to how datasets are split to avoid data leakage across patients. Among the provided secti...
🎯 Retrieved node IDs: ['0006', '0007']
📄 Sections found: ['7. Dataset Preparation', '8. Training and Evaluation']

📝 Answer:
Patient‑level separation is the practice of ensuring that images from the same patient never appear in both the training and test (or validation) sets; otherwise the evaluation can become unrealistically optimistic. (7. Dataset Preparation, p 7)


In [21]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What factors can affect the performance of an AI radiology model?",
    "Why is external validation important for medical AI?",
    "What are the steps involved in an AI-assisted radiology workflow, and what role does human review play?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What factors can affect the performance of an AI radiology model?
A: Factors that can influence how well an AI radiology model works include:

- **Network design** – the choice of convolutional filters, pooling/strided operations, and depth of the CNN determines what visual patterns the model can capture (Section ‘6. Convolutional Neural Networks’, p. 6).  
- **Trans...
-------------------------------------------------------

Q: Why is external validation important for medical AI?
A: External validation is crucial because it tests a model on data from a different institution or acquisition environment, providing insight into how well the system generalizes beyond the training set (8. Training and Evaluation, p. 8). It also guards against performance drops caused by distribution ...
-------------------------------------------------------

Q: What are the steps involved in an AI-assisted radiology workflow, and what role does human review play?
A: The AI‑assisted radiology workflow p